# Python for AI — A Complete RAG Chatbot
## Ask questions about your own `.txt` and PDF files,

 FILES (.txt / .pdf)
      |
   1. LOAD      read the text, keep source + page
      |
   2. CHUNK     cut into small pieces (paragraphs, then sentences)
      |
   3. STORE     embed each chunk and save it in Chroma
      |
 ------------------------------  (all of the above happens once)
      |
 YOUR QUESTION
      |
   4. RETRIEVE  Chroma finds the chunks closest in meaning
      |
   5. ANSWER    Groq answers using ONLY those chunks (and remembers the chat)
      |
   ANSWER + SOURCES (file name and page)
```

**RAG** = **R**etrieval **A**ugmented **G**eneration: *retrieve* the right chunks, then *generate* the answer from them.


In [1]:
!pip install groq chromadb pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 106.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 93.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 96.5 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      Su

In [2]:
from groq import Groq
from google.colab import userdata
api_key=userdata.get('areeba')
Client=Groq(api_key=api_key)
MODEL="openai/gpt-oss-120b"

In [3]:
import os
os.makedirs("docs",exist_ok=True)

In [4]:
%%writefile docs/faq.txt
Welcome to Travel Agency.

We offer domestic and international travel packages to popular destinations. Customers can book flight tickets, hotel accommodations, holiday packages, and guided tours.

Our travel packages range from 3 to 14 days, depending on the destination. Prices vary based on the destination, travel dates, hotel category, and number of travelers.

Customers must provide a valid passport for international travel and any required visas. Domestic travelers must carry a valid government-issued identification document.

Bookings can be made by paying a 30 percent advance payment. The remaining balance must be paid at least 7 days before departure.

Customers can cancel their booking up to 14 days before departure and receive a refund, subject to the applicable cancellation charges. Cancellations made less than 14 days before departure may be non-refundable.

Travelers are responsible for checking passport validity, visa requirements, baggage allowances, and other travel regulations before departure.

For questions or bookings, email support@sampletravel.example. Our office is open Monday to Friday, 10 am to 5 pm.

Thank you for choosing Sample Travel Agency. We look forward to making your journey comfortable and memorable.

Writing docs/faq.txt


In [6]:
print(os.listdir("docs"))

['faq.txt', 'Travel Agency Marketing Strategy.pdf']


In [15]:
import os
import re
from pypdf import PdfReader

# STEP 01: DOCUMENT LOADING / INGESTION

def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return [{
            "text": f.read(),
            "source": os.path.basename(path),
            "page": 1
        }]


def load_pdf(path):
    pages = []
    reader = PdfReader(path)

    for number, page in enumerate(reader.pages, start=1):
        pages.append({
            "text": page.extract_text() or "",
            "source": os.path.basename(path),
            "page": number
        })

    return pages


def load_folder(folder):
    pages = []

    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)

        if not os.path.isfile(path):
            continue

        if name.lower().endswith(".txt"):
            pages += load_txt(path)

        elif name.lower().endswith(".pdf"):
            pages += load_pdf(path)

    return pages


# STEP 02: CLEANING AND CHUNKING

def clean_text(text):
    return " ".join(text.split())


def chunk_sentences(text, max_chars=400):
    """Pack sentences into chunks of approximately max_chars."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    chunks = []
    current = ""

    for sentence in sentences:
        if not sentence:
            continue

        if current and len(current) + 1 + len(sentence) > max_chars:
            chunks.append(current)
            current = sentence
        else:
            current = (current + " " + sentence).strip()

    if current:
        chunks.append(current)

    return chunks


def chunk_paragraphs(text, max_chars=400):
    """Split at blank lines, then chunk each paragraph."""
    chunks = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = clean_text(paragraph)

        if paragraph:
            chunks.extend(chunk_sentences(paragraph, max_chars))

    return chunks


def build_chunks(folder, max_chars=400):
    all_chunks = []

    for page in load_folder(folder):
        if not page["text"].strip():
            print(
                "Skipped empty page (possibly scanned; OCR may be needed):",
                page["source"],
                "page",
                page["page"]
            )
            continue

        for piece in chunk_paragraphs(page["text"], max_chars):
            all_chunks.append({
                "text": piece,
                "source": page["source"],
                "page": page["page"]
            })

    return all_chunks


# STEP 03: RUN THE PIPELINE

chunks = build_chunks("docs", max_chars=400)

print(len(chunks), "chunks")

if len(chunks) > 60:
    print(chunks[60])
else:
    print(" Available chunks:", len(chunks))


63 chunks
{'text': 'Travelers are responsible for checking passport validity, visa requirements, baggage allowances, and other travel regulations before departure.', 'source': 'faq.txt', 'page': 1}


In [16]:
#STEP 04: EMBEDDING and STORING CHUNKS IN CHROMADB
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embed=DefaultEmbeddingFunction()
chroma=chromadb.Client()

def store_chunks(chunks,name="docs_kb"):
  try:
    chroma.delete_collection(name)
  except Exception:
      pass
  collection=chroma.create_collection(name=name,embedding_function=embed)
  collection.upsert(
      documents=[c["text"] for c in chunks],
      metadatas=[{"source":c["source"],"page":c["page"]} for c in chunks],
      ids=[f"chunk{i}" for i in range(len(chunks))]
  )
  return collection
collection=store_chunks(chunks)
print("Stored:",collection.count(),"chunks")


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 31.0MiB/s]


Stored: 63 chunks


In [17]:
#STEP 05: RETRIEVE
def retrieve(question,n=3):
  result=collection.query(query_texts=[question],n_results=min(n,collection.count()))
  found=[]
  for text,meta,dist in zip(result["documents"][0],result["metadatas"][0],result["distances"][0]):
    found.append({"text":text,"source":meta["source"],"page":meta["page"],"distance":dist})
    return found

for item in retrieve("What is Retrieval Augmented Generation?"):
  print(f'{item["distance"]:.2f} | {item["source"]} page {item["page"]} | {item["text"][:70]}...')

1.46 | Travel Agency Marketing Strategy.pdf page 4 | Search engine optimization (SEO) helps improve the website's visibilit...


In [18]:
#STEP 06: ANSWER WITH MEMORY
RULES=("You answer questions using ONLY the context the user gives you. "
       "If the answer is not in the context, reply exactly: I don't know.")
def ask(question,found,history):
  context="\n\n".join(f'[{c["source"]},page{c["page"]}]\n{c["text"]}' for c in found)
  messages=[{"role":"system","content":RULES}]
  messages+=history
  messages.append({"role":"user","content":f"Context:\n{context}\n\nQuestion:{question}"})
  response=Client.chat.completions.create(
      model=MODEL,
      messages=messages,
      temperature=0,
      max_tokens=400
  )
  return response.choices[0].message.content
chat_history=[]

def rag_chat(question):
  found=retrieve(question)
  reply=ask(question,found,chat_history[-6:])
  chat_history.append({"role":"user","content":question})
  chat_history.append({"role":"assistant","content":reply})
  return reply,found

def show(question):
  reply,found=rag_chat(question)
  print("Q: ",question)
  print("A: ",reply)
  print("Sources:",",".join(f"{c["source"]} p{c["page"]}" for c in found))
  print()

In [19]:
def chatbot():
    print("Welcome to Travel agent Chatbot!")
    print("Ask me anything. Type 'exit' to stop.\n")

    while True:
        question = input("You: ").strip()

        # Stop the loop when the user types exit
        if question.lower() == "exit":
            print("Bot: Goodbye! Have a nice day.")
            break

        # Handle empty input
        if not question:
            print("Bot: Please enter a question.")
            continue

        # Get the model's answer
        answer = show(question)

        if answer is not None:
            print("Bot:", answer)

# Start the chatbot
chatbot()

Welcome to Travel agent Chatbot!
Ask me anything. Type 'exit' to stop.

You: hi \
Q:  hi \
A:  I don't know.
Sources: faq.txt p1

You: i want to know about strategies
Q:  i want to know about strategies
A:  Based on the provided context, the travel agency’s strategies focus on:

1. **Building a strong brand image** – positioning the agency as a trusted choice for a wide range of travelers (individuals, families, students, couples, and corporate clients).  
2. **Increasing brand awareness** – using both traditional (e.g., print ads, radio, events) and digital marketing channels (e.g., social media, email campaigns, online ads) to reach potential travelers.  
3. **Attracting new customers** – targeting the different traveler segments with tailored messages that highlight the agency’s reliability and value.  
4. **Improving sales** – converting the increased awareness and new customer interest into bookings through coordinated marketing efforts across the chosen channels.
Sources: Travel 